## ------------------------------------------------------------ ###
## Set up environment
## ------------------------------------------------------------ ###

In [2]:
import os
import sys
import glob
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../ProcessEvents/')
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR # MOLLY_DIR_FF, RAINFALL_CSV_DIR, ENSEMBLE_MEMBERS, , CATCHMENTS
event_details_fp = 'EventDetails' # 'EventDetails_v5'
flood_5km_results_fp = "5km_total" # "5km_total_v5"

## ------------------------------------------------------------ ###
## Get list of catchments with all outputs
## ------------------------------------------------------------ ###

In [7]:
tes = pd.read_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments_added_soilvars.pkl")

In [4]:
tes = pd.read_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments_added_soilvars_and_othervars.pkl")
tes

,max_precip,t_global,t_local,x_idx,y_idx,x_idx_global,y_idx_global,x_coord,y_coord,ens,...,lu_at_peak,hc_at_peak.1,peak_cell_slope_avg,peak_cell_slope_max,catchment_slope_avg,catchment_slope_max,peak_cell_sink_frac,water_prop,urban_prop,inside_prop
0,33.823380,4922,24,11,3,67,11,137500.0,22500.0,1,...,0.048631,0.600732,NaN,NaN,NaN,NaN,NaN,0.021324,0.036999,54.422005
1,93.955116,5897,7,27,18,83,26,217500.0,97500.0,1,...,0.036350,0.330241,NaN,NaN,NaN,NaN,NaN,0.033366,0.029847,87.087601
2,32.004837,4836,46,26,16,82,24,212500.0,87500.0,1,...,0.038800,0.380427,NaN,NaN,NaN,NaN,NaN,0.011512,0.018142,88.767998
3,57.549026,6110,10,17,6,73,14,167500.0,37500.0,1,...,0.034814,0.322188,NaN,NaN,NaN,NaN,NaN,0.015873,0.128260,62.273605
4,32.734352,4774,9,17,7,73,15,167500.0,42500.0,1,...,0.028875,0.253981,NaN,NaN,NaN,NaN,NaN,0.020076,0.334944,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105584,35.923145,5175,22,8,6,118,48,392500.0,207500.0,15,...,0.020267,0.103449,NaN,NaN,NaN,NaN,NaN,0.008820,0.033072,100.000000
105585,47.853752,6812,2,4,11,114,53,372500.0,232500.0,15,...,0.019731,0.097438,NaN,NaN,NaN,NaN,NaN,0.029844,0.046392,100.000000
105586,36.347977,5650,1,7,4,117,46,387500.0,197500.0,15,...,0.020276,0.107064,NaN,NaN,NaN,NaN,NaN,0.028684,0.055757,71.051201
105587,30.292673,7799,8,5,7,115,49,377500.0,212500.0,15,...,0.019551,0.095549,NaN,NaN,NaN,NaN,NaN,0.066281,0.068747,94.345596


In [15]:
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())

catchments_with_flood_output = []
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/{catchment_name}.pkl"
    fp2 = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/{catchment_name}_new.pkl"
    flood_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/{flood_5km_results_fp}/Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp) and (os.path.isfile(fp) or os.path.isfile(fp2)):
        catchments_with_flood_output.append(catchment_num)
    else:
        pass
        #print(catchment_num)
        #print(os.path.isfile(fp))
        #print(os.path.isfile(flood_fp))
        
# catchments_with_flood_output  = ['40', '23', '105']     

## ------------------------------------------------------------ ###
## Get list of catchments with all outputs
## ------------------------------------------------------------ ###

In [60]:
rainfall_events_all = []
for catchment_num in catchments_with_flood_output:
    if catchment_num not in ['890']:
        catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
        fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/all_events_added_soilvars_withbias.csv"
        rainfall_events = pd.read_csv(fp)
        if 'lu_at_peak.2' in rainfall_events.columns:
            print("YES", catchment_num)
    #     if 'start_month' in rainfall_events.columns:
    #         print("YES", catchment_num)
    #         del rainfall_events['start_day']
    #         del rainfall_events['start_hour']
    #         rainfall_events.rename(columns={'start_month': 'month'}, inplace=True)
    #         rainfall_events.rename(columns={'start_year':'start_year'}, inplace=True)
    #         rainfall_events.rename(columns={'start_day':'day'}, inplace=True)
        #rainfall_events_complete = rainfall_events[rainfall_events['mismatch']!=True].copy()
        rainfall_events['catchment_num'] = catchment_num
        if len(rainfall_events) ==0:
            print(catchment_num)
        rainfall_events_all.append(rainfall_events) 
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   
del rainfall_events_all_df['hc_at_peak.1']
len(rainfall_events_all_df.loc[~rainfall_events_all_df["catchment_num"].map(lambda x: isinstance(x, (int, float))), "catchment_num"].unique())

114

## ------------------------------------------------------------ ###
## Check if any catchments have NANs for any variables
## ------------------------------------------------------------ ###

In [62]:
nulls = rainfall_events_all_df[rainfall_events_all_df['hc_at_peak'].isnull()]
np.unique(nulls)

array([], dtype=object)

In [63]:
nulls = rainfall_events_all_df[rainfall_events_all_df['fu_at_peak_new'].isnull()][['catchment_num', 'ens', 'fu_at_peak_new', 'lu_at_peak', 'event_num']]
nulls

,catchment_num,ens,fu_at_peak_new,lu_at_peak,event_num


## ------------------------------------------------------------ ###
## Check whether any grid cells have multiple events in them
## ------------------------------------------------------------ ###

In [64]:
dupes = rainfall_events_all_df[rainfall_events_all_df.duplicated(subset=["x_coord", "y_coord"],
        keep=False)].sort_values(["x_coord", "y_coord"])

len(dupes[['month', 'lu_at_peak', 'max_precip', 'ens', "x_coord", "y_coord"]])

106430

## ------------------------------------------------------------ ###
## Save to file
## ------------------------------------------------------------ ###

In [65]:
rainfall_events_all_df.to_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments_added_soilvars.pkl")